# Strands Agents in SageMaker with DataFrame State

**Track:** Applied Agent Engineering Foundations  
**Module fit:** M4 — Agentic AI: Design → Prototype → Production  
**Delivery intent:** classroom-ready Strands notebook for SageMaker

## Why this notebook exists
You asked that the agent notebook in M4 should use **Strands Agents** and should avoid Azure and backend JSON persistence.

This notebook therefore uses:
- Strands Agents
- Amazon Bedrock as the model provider
- S3 read-only policy context
- in-notebook pandas DataFrames for action state and trace state


## Learning goals
1. Define Strands tools with `@tool`
2. Give the agent both **knowledge tools** and **action tools**
3. Keep state in DataFrames instead of a backend JSON file
4. Inspect tool usage for traceability
5. Build a mini hack around a real enterprise workflow


In [ ]:

# Uncomment if needed.
# %pip install -q boto3 pandas sentence-transformers faiss-cpu strands-agents strands-agents-tools


## Step 1 — Configuration

Change only the config cell first.
If your HR documents already exist in S3, set `USE_S3 = True`.


In [ ]:

from __future__ import annotations

import io
import os
from pathlib import Path

import boto3
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
BEDROCK_MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"

USE_S3 = False
S3_BUCKET = "replace-me"
S3_PREFIX = "replace-me/hr_docs/"

TOP_K = 3
LOCAL_WORK_DIR = Path("./m4_strands_agent_artifacts")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3", region_name=AWS_REGION)


## Step 2 — Load small HR knowledge context

This is the read-only knowledge source for the agent.
The agent will use a retrieval tool over this corpus.


In [ ]:

def list_text_keys(bucket: str, prefix: str) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".txt") or key.endswith(".md"):
                keys.append(key)
    return keys

def read_s3_text(bucket: str, key: str) -> str:
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8", errors="ignore")

if USE_S3:
    hr_docs = [{"source": key, "text": read_s3_text(S3_BUCKET, key)} for key in list_text_keys(S3_BUCKET, S3_PREFIX)]
else:
    hr_docs = [
        {"source": "leave_policy.txt", "text": "Employees can apply casual leave and earned leave. Planned leave should be requested in advance."},
        {"source": "approval_policy.txt", "text": "Managers may approve or reject leave depending on staffing needs and leave balance."},
        {"source": "ticket_policy.txt", "text": "HR support tickets should include employee id, priority, and issue summary."},
    ]

hr_docs_df = pd.DataFrame(hr_docs)
display(hr_docs_df)


## Step 3 — Build a local retrieval helper

We use local embeddings for the retrieval tool so the agent can search policy context quickly and cheaply.


In [ ]:

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_rows = []
for row in hr_docs:
    chunk_rows.append({
        "source": row["source"],
        "chunk_id": f'{row["source"]}::001',
        "chunk_text": row["text"]
    })

knowledge_df = pd.DataFrame(chunk_rows)
knowledge_embeddings = embedding_model.encode(
    knowledge_df["chunk_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False
)

index = faiss.IndexFlatIP(knowledge_embeddings.shape[1])
index.add(np.array(knowledge_embeddings, dtype="float32"))

display(knowledge_df)


## Step 4 — Define DataFrame-backed workflow state

This replaces file-based backend JSON for the classroom.


In [ ]:

leave_requests_df = pd.DataFrame(columns=["employee_id", "days", "reason", "status", "created_at"])
tool_trace_df = pd.DataFrame(columns=["tool_name", "tool_input", "tool_output", "timestamp"])

def log_tool(tool_name: str, tool_input: str, tool_output: str):
    global tool_trace_df
    tool_trace_df = pd.concat([
        tool_trace_df,
        pd.DataFrame([{
            "tool_name": tool_name,
            "tool_input": tool_input,
            "tool_output": tool_output,
            "timestamp": pd.Timestamp.utcnow()
        }])
    ], ignore_index=True)


## Step 5 — Create Strands tools

Strands tools are plain Python functions decorated with `@tool`.


In [ ]:

from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator, current_time

@tool
def search_hr_policy(query: str) -> str:
    """Search the HR policy corpus and return the top matching snippets."""
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb, dtype="float32"), TOP_K)

    rows = []
    for score, idx_ in zip(scores[0], idxs[0]):
        row = knowledge_df.iloc[int(idx_)]
        rows.append(f"[score={float(score):.4f}] {row['source']}: {row['chunk_text']}")

    output = "\n\n".join(rows)
    log_tool("search_hr_policy", query, output)
    return output

@tool
def apply_leave(employee_id: str, days: int, reason: str) -> str:
    """Apply leave for an employee and store the request in a DataFrame."""
    global leave_requests_df
    new_row = pd.DataFrame([{
        "employee_id": employee_id,
        "days": int(days),
        "reason": reason,
        "status": "submitted",
        "created_at": pd.Timestamp.utcnow()
    }])

    leave_requests_df = pd.concat([leave_requests_df, new_row], ignore_index=True)
    output = f"Leave request submitted for {employee_id} for {days} day(s). Reason: {reason}"
    log_tool("apply_leave", f"{employee_id}|{days}|{reason}", output)
    return output

@tool
def get_leave_summary(employee_id: str) -> str:
    """Return leave history summary for an employee from the in-memory DataFrame."""
    employee_rows = leave_requests_df.loc[leave_requests_df["employee_id"] == employee_id]

    if len(employee_rows) == 0:
        output = f"No leave requests found for {employee_id}."
        log_tool("get_leave_summary", employee_id, output)
        return output

    total_days = int(employee_rows["days"].astype(int).sum())
    output = f"{employee_id} has {len(employee_rows)} request(s) totaling {total_days} day(s) of leave."
    log_tool("get_leave_summary", employee_id, output)
    return output


## Step 6 — Build the Strands agent


In [ ]:

bedrock_model = BedrockModel(
    model_id=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.2,
)

agent = Agent(
    model=bedrock_model,
    tools=[current_time, calculator, search_hr_policy, apply_leave, get_leave_summary],
    system_prompt=(
        "You are an HR operations agent. "
        "Use tools whenever policy lookup or action execution is needed. "
        "Do not invent policy details. "
        "When a leave action is requested, call the relevant tool."
    )
)

print("Agent ready.")


## Step 7 — Try the agent

Suggested classroom prompts:
- `What does the leave policy say about planned leave?`
- `Apply 2 days leave for EMP001 because of fever and then show the leave summary.`
- `What time is it and how many leave days were just applied?`


In [ ]:
result = agent("Apply 2 days leave for EMP001 because of fever, then check the leave summary and mention any relevant policy guidance.")
print(getattr(result, "message", result))


## Step 8 — Inspect state and trace tables

This is the notebook-native observability layer for the action flow.


In [ ]:

display(leave_requests_df)
display(tool_trace_df)


## Step 9 — Optional local persistence

Still no S3 writes. Only local notebook artifacts if you want them.


In [ ]:

leave_requests_df.to_csv(LOCAL_WORK_DIR / "leave_requests.csv", index=False)
tool_trace_df.to_csv(LOCAL_WORK_DIR / "tool_trace.csv", index=False)

print("Saved local agent artifacts:")
for p in sorted(LOCAL_WORK_DIR.glob("*")):
    print("-", p.name)


## Mini-hack — M4 Strands challenge

Learners can extend the agent in one of these ways:
1. Add a tool that creates an HR ticket DataFrame entry.
2. Add a policy guard that blocks leave requests above a threshold.
3. Add a manager-approval tool.
4. Add a trace summary tool that reports how many tools the agent used.

## Reflection questions
- What changed when the workflow moved from simple RAG to an agent?
- Which parts are knowledge retrieval, and which parts are action execution?
- Why is DataFrame state acceptable for a classroom lab but not enough for production?


## Instructor wrap-up

This notebook demonstrates the agentic pattern cleanly:
**reason → choose tool → execute action → keep trace → respond**.

That makes it a strong bridge from your M2 RAG labs into M4 agent design and production thinking.
